In [ ]:
# Data Loading and Exploration

print("📊 Loading training data...")
train_df = pd.read_csv(TRAIN_PATH)

print(f"Shape: {train_df.shape}")
print(f"\nColumns: {train_df.columns.tolist()}")
print(f"\nFirst few rows:")
print(train_df.head())

# Check for required columns
if "comment_text" not in train_df.columns:
    print("\n❌ 'comment_text' column not found!")
    print("Available columns:", train_df.columns.tolist())
else:
    print("\n✅ 'comment_text' column found")

if "toxic" not in train_df.columns:
    print("\n❌ 'toxic' column not found!")
    print("Available columns:", train_df.columns.tolist())
else:
    print("\n✅ 'toxic' column found")
    
# Data statistics
print(f"\n📈 Data Statistics:")
print(f"Total comments: {len(train_df)}")
print(f"Toxic comments: {train_df['toxic'].sum()}")
print(f"Non-toxic comments: {len(train_df) - train_df['toxic'].sum()}")
print(f"Toxicity rate: {train_df['toxic'].mean():.2%}")

# Visualize class distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
train_df['toxic'].value_counts().plot(kind='bar', ax=ax1, color=['green', 'red'])
ax1.set_title('Class Distribution', fontsize=14, fontweight='bold')
ax1.set_xlabel('Label (0=Non-toxic, 1=Toxic)')
ax1.set_ylabel('Count')
ax1.set_xticklabels(['Non-toxic', 'Toxic'], rotation=0)

# Pie chart
train_df['toxic'].value_counts().plot(kind='pie', ax=ax2, autopct='%1.1f%%', 
                                       colors=['green', 'red'], labels=['Non-toxic', 'Toxic'])
ax2.set_title('Class Proportion', fontsize=14, fontweight='bold')
ax2.set_ylabel('')

plt.tight_layout()
plt.show()

# Text length analysis
train_df['text_length'] = train_df['comment_text'].astype(str).str.len()
train_df['word_count'] = train_df['comment_text'].astype(str).str.split().str.len()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Character length by toxicity
train_df.groupby('toxic')['text_length'].hist(ax=ax1, alpha=0.7, bins=50, label=['Non-toxic', 'Toxic'])
ax1.set_title('Character Length Distribution', fontsize=14, fontweight='bold')
ax1.set_xlabel('Character Length')
ax1.set_ylabel('Frequency')
ax1.legend(['Non-toxic', 'Toxic'])

# Word count by toxicity
train_df.groupby('toxic')['word_count'].hist(ax=ax2, alpha=0.7, bins=50, label=['Non-toxic', 'Toxic'])
ax2.set_title('Word Count Distribution', fontsize=14, fontweight='bold')
ax2.set_xlabel('Word Count')
ax2.set_ylabel('Frequency')
ax2.legend(['Non-toxic', 'Toxic'])

plt.tight_layout()
plt.show()

print(f"\n📏 Text Length Statistics:")
print(train_df.groupby('toxic')[['text_length', 'word_count']].describe())

In [ ]:
# Data Preparation

# Clean data
print("🧹 Preparing data...")
train_df = train_df[["comment_text", "toxic"]].copy()
train_df["toxic"] = train_df["toxic"].astype(float)

# Remove any NaN values
initial_len = len(train_df)
train_df = train_df.dropna()
print(f"Removed {initial_len - len(train_df)} rows with missing values")

# Split into train and validation
dataset = Dataset.from_pandas(train_df)
dataset = dataset.train_test_split(
    test_size=config["data"]["validation_split"],
    seed=config["data"]["random_seed"]
)

train_ds = dataset["train"]
val_ds = dataset["test"]

print(f"\n✅ Dataset split complete:")
print(f"   Training: {len(train_ds)} samples")
print(f"   Validation: {len(val_ds)} samples")

# Load tokenizer
print(f"\n🔤 Loading tokenizer: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Tokenization function
def preprocess(batch):
    return tokenizer(
        batch["comment_text"],
        truncation=True,
        padding="max_length",
        max_length=config["model"]["max_length"]
    )

# Tokenize datasets
print("🔄 Tokenizing datasets...")
train_ds = train_ds.map(preprocess, batched=True)
val_ds = val_ds.map(preprocess, batched=True)

# Rename label column
train_ds = train_ds.rename_column("toxic", "labels")
val_ds = val_ds.rename_column("toxic", "labels")

# Set format for PyTorch
train_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
val_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

print("✅ Tokenization complete")

# Show sample
print("\n📄 Sample tokenized input:")
sample = train_ds[0]
print(f"Text: {tokenizer.decode(sample['input_ids'])}")
print(f"Label: {sample['labels'].item()}")

In [ ]:
# Model Training

# Load model
print("🤖 Loading model...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=config["model"]["num_labels"]
)
print(f"✅ Model loaded: {MODEL_NAME}")

# Metrics function
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    logits = np.squeeze(logits)
    labels = labels.astype(int)
    probs = 1 / (1 + np.exp(-logits))  # sigmoid
    
    return {
        "roc_auc": roc_auc_score(labels, probs),
        "pr_auc": average_precision_score(labels, probs),
    }

# Training arguments
training_args = TrainingArguments(
    output_dir="./toxicity_model",
    per_device_train_batch_size=config["training"]["batch_size"],
    per_device_eval_batch_size=config["training"]["batch_size"],
    num_train_epochs=config["training"]["epochs"],
    learning_rate=config["training"]["learning_rate"],
    weight_decay=config["training"]["weight_decay"],
    warmup_steps=config["training"]["warmup_steps"],
    logging_steps=config["training"]["logging_steps"],
    eval_steps=config["training"]["eval_steps"],
    save_steps=config["training"]["save_steps"],
    evaluation_strategy="steps",
    save_strategy="steps",
    load_best_model_at_end=True,
    metric_for_best_model="roc_auc",
    fp16=config["training"]["fp16"],
    report_to="none"
)

# Create trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

# Train
print("\n🚀 Starting training...")
print(f"Epochs: {config['training']['epochs']}")
print(f"Batch size: {config['training']['batch_size']}")
print(f"Learning rate: {config['training']['learning_rate']}")
print("\nThis may take a while...\n")

trainer.train()

print("\n✅ Training completed!")

In [ ]:
# Model Evaluation

print("📊 Evaluating model on validation set...")
val_metrics = trainer.evaluate()

print("\n✅ Validation Metrics:")
for key, value in val_metrics.items():
    if key.startswith("eval_"):
        metric_name = key.replace("eval_", "").upper()
        print(f"   {metric_name}: {value:.4f}")

# Get predictions for detailed analysis
print("\n🔮 Generating predictions...")
preds = trainer.predict(val_ds)
logits = np.squeeze(preds.predictions)
labels = preds.label_ids.astype(int)
probs = 1 / (1 + np.exp(-logits))

print(f"✅ Generated {len(probs)} predictions")

In [ ]:
# Threshold Tuning with Visualizations

print("🎯 Finding optimal decision threshold...")

thresholds = np.linspace(
    config["threshold"]["range_start"],
    config["threshold"]["range_end"],
    config["threshold"]["num_thresholds"]
)

f1_scores = []
precision_scores = []
recall_scores = []

for t in thresholds:
    predictions = (probs >= t).astype(int)
    f1 = f1_score(labels, predictions)
    f1_scores.append(f1)
    
    # Calculate precision and recall
    from sklearn.metrics import precision_score, recall_score
    prec = precision_score(labels, predictions, zero_division=0)
    rec = recall_score(labels, predictions, zero_division=0)
    precision_scores.append(prec)
    recall_scores.append(rec)

# Find best threshold
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]
best_f1 = f1_scores[best_idx]

print(f"\n✅ Optimal Threshold: {best_threshold:.3f}")
print(f"✅ Best F1 Score: {best_f1:.4f}")

# Visualize threshold tuning
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# F1 vs Threshold
ax1.plot(thresholds, f1_scores, 'b-', linewidth=2, label='F1 Score')
ax1.axvline(best_threshold, color='red', linestyle='--', linewidth=2, label=f'Optimal: {best_threshold:.3f}')
ax1.axhline(best_f1, color='green', linestyle=':', alpha=0.5)
ax1.set_xlabel('Threshold', fontsize=12)
ax1.set_ylabel('F1 Score', fontsize=12)
ax1.set_title('F1 Score vs Threshold', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Precision-Recall curve
ax2.plot(recall_scores, precision_scores, 'g-', linewidth=2)
ax2.scatter(recall_scores[best_idx], precision_scores[best_idx], 
            color='red', s=200, zorder=5, label=f'Optimal (t={best_threshold:.3f})')
ax2.set_xlabel('Recall', fontsize=12)
ax2.set_ylabel('Precision', fontsize=12)
ax2.set_title('Precision-Recall Curve', fontsize=14, fontweight='bold')
ax2.legend ()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Confusion matrix at optimal threshold
print("\n📊 Confusion Matrix at optimal threshold:")
optimal_preds = (probs >= best_threshold).astype(int)
cm = confusion_matrix(labels, optimal_preds)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, 
            xticklabels=['Non-toxic', 'Toxic'],
            yticklabels=['Non-toxic', 'Toxic'])
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('Actual', fontsize=12)
ax.set_title('Confusion Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Classification report
print("\n📋 Classification Report:")
print(classification_report(labels, optimal_preds, target_names=['Non-toxic', 'Toxic']))

In [ ]:
# Save Model

print(f"💾 Saving model to {SAVE_DIR}...")
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

# Save optimal threshold
threshold_info = {
    "optimal_threshold": float(best_threshold),
    "f1_score": float(best_f1),
    "roc_auc": float(val_metrics["eval_roc_auc"]),
    "pr_auc": float(val_metrics["eval_pr_auc"])
}

with open(os.path.join(SAVE_DIR, "threshold_info.json"), "w") as f:
    json.dump(threshold_info, f, indent=2)

print(f"✅ Model saved successfully!")
print(f"✅ Threshold info saved to {SAVE_DIR}/threshold_info.json")
print(f"\n📌 Use threshold {best_threshold:.3f} in the Streamlit app for best results")

In [ ]:
# Sanity Check - Test Model

print("🧪 Running sanity check with sample comments...")

test_samples = [
    "You are a stupid idiot",
    "Thank you so much for your help",
    "Go die, nobody likes you",
    "This is a wonderful community",
    "Fuck you asshole",
    "I really appreciate your thoughtful response",
    "You're pathetic and useless",
    "Have a great day!"
]

device = model.device

print("\n" + "="*60)
for sample in test_samples:
    inputs = tokenizer(sample, return_tensors="pt", truncation=True, padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        logit = model(**inputs).logits.item()
        prob = 1 / (1 + torch.exp(-torch.tensor(logit))).item()
    
    label = "Toxic" if prob >= best_threshold else "Non-toxic"
    emoji = "🔴" if prob >= best_threshold else "🟢"
    
    print(f"{emoji} {prob:.3f} | {label:12s} | {sample}")

print("="*60)
print("\n✅ Sanity check complete!")

In [ ]:
!pip install captum

In [ ]:
import torch
import numpy as np
from captum.attr import IntegratedGradients
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# LOAD MODEL + TOKENIZER
MODEL_CHECKPOINT = "/kaggle/working/exported_model"
BASE_MODEL = "distilbert-base-uncased"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_CHECKPOINT)
model.to(device)
model.eval()

# FORWARD FUNCTION (EMBEDDINGS)
def forward_func(inputs_embeds, attention_mask):
    outputs = model(
        inputs_embeds=inputs_embeds,
        attention_mask=attention_mask
    )
    return outputs.logits.squeeze(-1)

ig = IntegratedGradients(forward_func)

# EXPLANATION FUNCTION
def explain_text(text, n_steps=50):
    encoded = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=128
    )

    input_ids = encoded["input_ids"].to(device)
    attention_mask = encoded["attention_mask"].to(device)

    # Get embeddings
    embeddings = model.distilbert.embeddings(input_ids)

    # Baseline = zero embedding
    baseline = torch.zeros_like(embeddings).to(device)

    # Integrated Gradients
    attributions = ig.attribute(
        inputs=embeddings,
        baselines=baseline,
        additional_forward_args=(attention_mask,),
        n_steps=n_steps
    )

    # Sum over embedding dimensions
    attributions = attributions.sum(dim=-1).squeeze(0)

    tokens = tokenizer.convert_ids_to_tokens(input_ids.squeeze(0))

    # Normalize for readability
    attributions = attributions / torch.norm(attributions)

    return tokens, attributions.detach().cpu().numpy()

# VISUALIZATION
def visualize_attributions(tokens, attributions):
    print("\nToken-level attributions:\n")
    for tok, score in zip(tokens, attributions):
        if tok in ["[CLS]", "[SEP]", "[PAD]"]:
            continue
        print(f"{tok:>12s} : {score:+.4f}")

# TEST
examples = [
    "You are a stupid idiot",
    "Thank you so much for your help",
    "Go die, nobody likes you",
    "This is a wonderful community"
]

for text in examples:
    print("=" * 80)
    print("TEXT:", text)
    tokens, scores = explain_text(text)
    visualize_attributions(tokens, scores)
